## Evaluation

Let’s evaluate our results to see how similar they are to the original experiment (from [here](https://github.com/philshiu/Drosophila_brain_model)). First, we define the same utils, then we check a few numbers and metrics.

Note: *Due to a random factor, the results will never be exactly the same.*

In [1]:
import pandas as pd
import numpy as np
from brian2 import ms

##########
# analysis
def load_exps(l_pqt):
    '''Load simulation results from disk

    Parameters
    ----------
    l_pkl : list
        List of parquet files with simulation results

    Returns
    -------
    exps : df
        data for all experiments 'path_res'
    '''
    # cycle through all experiments
    dfs = []
    for p in l_pqt:
        # load metadata from pickle
        with open(p, 'rb') as f:
            df = pd.read_parquet(p)
            df.loc[:, 't'] = df.loc[:, 't'].astype(float)
            dfs.append(df)

    df = pd.concat(dfs)

    return df

def get_rate(df, t_run, n_run, flyid2name=dict()):
    '''Calculate rate and standard deviation for all experiments
    in df

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe generated with `load_exps` containing spike times
    t_run : float
        Trial duration in seconds
    n_run : int
        Number of trials
    flyid2name : dict (optional)
        Mapping between flywire IDs and custom names

    Returns
    -------
    df_rate : pd.DataFrame
        Dataframe with average firing rates
    df_std : pd.DataFrame
        Dataframe with standard deviation of firing rates
    '''

    rate, std, flyid, exp_name = [], [], [], []

    for e, df_e in df.groupby('exp_name', sort=False):
        for f, df_f in df_e.groupby('flywire_id'):

            r = np.zeros(n_run)
            for t, df_t in df_f.groupby('trial'):
                r[int(t)] = len(df_t) / t_run

            rate.append(r.mean())
            std.append(r.std())
            flyid.append(f)
            exp_name.append(e)

    d = {
        'r' : rate,
        'std': std,
        'flyid' : flyid,
        'exp_name' : exp_name,
    }
    df = pd.DataFrame(d)
    
    df_rate = df.pivot_table(columns='exp_name', index='flyid', values='r')
    df_std = df.pivot_table(columns='exp_name', index='flyid', values='std')
    
    if flyid2name:
        df_rate.insert(loc=0, column='name', value=df_rate.index.map(flyid2name).fillna(''))
        df_std.insert(loc=0, column='name', value=df_rate.index.map(flyid2name).fillna(''))

    return df_rate, df_std


In [ ]:
# load data from disk
df_spike = load_exps(['./outputs/sugarR_reconstruct_experiment/20261112_171030.parquet'])
df_spike

In [3]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }

In [ ]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = get_rate(df_spike, t_run=1000*ms, n_run=30, flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR_reconstruct_experiment', ascending=False)

In [5]:
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_reconstruct_experiment"]
print(f'Rate for neuron {id_mn9} is {x}')

Rate for neuron 720575940660219265 is 83.6


In [ ]:
df_spike_refrence = load_exps(['./outputs/sugarR_reconstruct_experiment/refrence.parquet'])
df_rate_refrence, df_rate_std_refrence = get_rate(df_spike_refrence, t_run=1000*ms, n_run=30, flyid2name=flyid2name)
df_rate_refrence.sort_values('sugarR', ascending=False)
x = df_rate_refrence.loc[id_mn9, "sugarR"]
print(f'refrence Rate for neuron {id_mn9} is {x}')

refrence Rate for neuron 720575940660219265 is 93.26666666666667
